# ETL en Colab — Rendimiento estudiantil
**Extract + Transform** equivalente al query de Power Query, en Python/pandas.

Replica: dedupe · normalización de vacíos · estandarización de categóricas ·
limpieza de numéricas · parseo de fechas · reglas de rango · imputación por
mediana según género · `parental_education_clean`.

> **Diferencias intencionales frente al query de Power Query**
> 1. **Decimales:** pandas usa siempre el punto como separador, así que aquí
>    convertimos `,` → `.` (en tu Excel en español hacías lo contrario).
> 2. **Vacíos:** todo va a `NaN` real (no a `""`), más limpio para analizar.
> 3. **Outliers:** aplicamos el set completo de reglas de rango **antes** de
>    imputar, para que las medianas no queden contaminadas por valores como
>    999 o 250 (tu query solo anulaba −3.5 y −5).


## 1. Extract — cargar la base

In [ ]:
import numpy as np
import pandas as pd
import re

# En Colab: al ejecutar se abre el botón para subir el CSV.
# Fuera de Colab, usa la ruta local del archivo.
try:
    from google.colab import files
    subido = files.upload()
    PATH = next(iter(subido))
except Exception:
    PATH = "student_performance_dirty.csv"

# dtype=str + keep_default_na=False -> nada se convierte solo;
# controlamos nosotros los vacíos y los tipos.
df = pd.read_csv(PATH, dtype=str, keep_default_na=False)
print("Carga:", df.shape)
df.head()

## 2. Limpieza de espacios y caracteres de control
Equivale a `Text.Trim` + `Text.Clean`.

In [ ]:
for c in df.columns:
    df[c] = (df[c].astype(str)
                  .str.replace(r"[\x00-\x1f\x7f]", "", regex=True)  # Text.Clean
                  .str.strip())                                        # Text.Trim

## 3. Duplicados exactos
`Table.Distinct` sobre la tabla cruda.

In [ ]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados exactos eliminados: {antes - len(df)}  ->  {df.shape}")

## 4. Vacíos → `NaN`
Un mismo "vacío" viene como `NA`, `N/A`, `null`, `NaN`, `-`, `?`, `sin dato`, etc.
Los pasamos a `NaN` en todas las columnas.

**Ojo:** `None` es token de vacío *y* categoría válida de `parental_education`
(significa "sin educación"), así que lo protegemos en esa columna.

In [ ]:
NULL_TOKENS = {"", "na", "n/a", "null", "nan", "-", "?", "sin dato", "none"}

def nullify(s, keep_none=False):
    toks = NULL_TOKENS - ({"none"} if keep_none else set())
    return s.mask(s.str.lower().isin(toks), np.nan)

for c in df.columns:
    df[c] = nullify(df[c], keep_none=(c == "parental_education"))

df.isna().sum()

## 5. Categóricas — `gender`
→ {Male, Female}

In [ ]:
gmap = {"male":"Male","m":"Male","hombre":"Male","masculino":"Male",
        "female":"Female","f":"Female","mujer":"Female","femenino":"Female"}
df["gender"] = df["gender"].str.lower().map(gmap).fillna(df["gender"])
print(sorted(df["gender"].dropna().unique()))

## 6. `student_id`
Quita el prefijo `S-` y convierte a entero.

In [ ]:
df["student_id"] = df["student_id"].str.replace("S-", "", regex=False).str.strip()
df["student_id"] = pd.to_numeric(df["student_id"], errors="coerce").astype("Int64")

## 7. Booleanas → {Yes, No}
`Internet_Access`, `extracurricular_activities`, `part_time_job`.

In [ ]:
YES = {"yes","y","1","true","sí","si"}
NO  = {"no","n","0","false"}

def yesno(s):
    low = s.str.lower()
    out = pd.Series(np.nan, index=s.index, dtype=object)
    out = out.mask(low.isin(YES), "Yes")
    out = out.mask(low.isin(NO),  "No")
    return out

for c in ["Internet_Access", "extracurricular_activities", "part_time_job"]:
    df[c] = yesno(df[c])
df[["Internet_Access","extracurricular_activities","part_time_job"]].apply(lambda s: s.value_counts(dropna=False))

## 8. Numéricas guardadas como texto
Quita `%`, cambia `,` → `.` y convierte a número.
Lo que no sea numérico (`NA`, etc.) queda en `NaN`.

In [ ]:
NUM = ["study_time_hours","attendance_percent","sleep_hours",
       "previous_grade","final_exam_score"]

for c in NUM:
    df[c] = (df[c].astype(str)
                  .str.replace("%", "", regex=False)
                  .str.replace(",", ".", regex=False)
                  .str.strip())
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[NUM].dtypes

## 9. `final_grade` → {A, B, C, D, F}

In [ ]:
df["final_grade"] = df["final_grade"].str.upper().str.strip()
print(sorted(df["final_grade"].dropna().unique()))

## 10. Fechas (`enrollment_date`)
Los formatos vienen mezclados, así que probamos varios en orden y lo que no
cuadra (fechas imposibles, texto) va a `NaT`. También normalizamos el
`" de "` de las fechas en español.

In [ ]:
FMTS = ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y",
        "%d/%m/%y", "%B %d, %Y", "%d %B %Y", "%Y%m%d"]

def parse_date(v):
    if pd.isna(v):
        return pd.NaT
    s = re.sub(r"\s+[Dd]e\s+", " ", str(v).strip())   # "28 de October de 2023" -> "28 October 2023"
    if s == "":
        return pd.NaT
    for f in FMTS:
        try:
            return pd.to_datetime(s, format=f)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df["enrollment_date_clean"] = df["enrollment_date"].apply(parse_date)
df = df.drop(columns=["enrollment_date"])
print("Fechas inválidas/vacías (NaT):", df["enrollment_date_clean"].isna().sum())

## 11. Duplicados cercanos
Tras estandarizar, las filas que solo diferían por mayúsculas ya son idénticas.

In [ ]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Casi-duplicados eliminados: {antes - len(df)}  ->  {df.shape}")

## 12. Reglas de rango (outliers → `NaN`)
Anulamos los valores fuera de dominio **antes** de imputar, para no ensuciar
las medianas.

In [ ]:
BOUNDS = {"study_time_hours": (0, 12), "attendance_percent": (0, 100),
          "sleep_hours": (0, 12), "previous_grade": (0, 100),
          "final_exam_score": (0, 100)}

for c, (lo, hi) in BOUNDS.items():
    fuera = ((df[c] < lo) | (df[c] > hi)).sum()
    df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
    print(f"{c}: {fuera} fuera de rango -> NaN")

## 13. Imputación por mediana según género
Rellena los `NaN` numéricos con la mediana del grupo (Female / Male),
igual que en tu query.

In [ ]:
for c in NUM:
    df[c] = df.groupby("gender")[c].transform(lambda s: s.fillna(s.median()))

df[NUM].isna().sum()

## 14. `parental_education_clean`
Mapeo a las 5 categorías canónicas.

In [ ]:
def clean_edu(v):
    if pd.isna(v):
        return np.nan
    e = str(v).lower().strip()
    if e in {"none","ninguno","sin educación","sin educacion"}: return "None"
    if e in {"high school","highschool","hs","secundaria","bachillerato"}: return "High School"
    if e in {"bachelor","bachelors","pregrado","universitario"}: return "Bachelors"
    if e in {"master","masters","msc","maestría","maestria","postgrado"}: return "Masters"
    if e in {"phd","ph.d.","doctorate","doctorado"}: return "PhD"
    return v

df["parental_education_clean"] = df["parental_education"].apply(clean_edu)
print(sorted(df["parental_education_clean"].dropna().unique()))

## 15. (Opcional) Duplicados lógicos por `student_id`
conservar una sola fila por estudiante (la matrícula más reciente).

In [ ]:
 df = (df.sort_values("enrollment_date_clean", ascending=False)
        .drop_duplicates(subset="student_id", keep="first")
        .reset_index(drop=True))
print("IDs repetidos restantes:", df["student_id"].duplicated().sum())

## 16. Reordenar columnas

In [ ]:
ORDER = ["student_id","gender","parental_education","parental_education_clean",
         "Internet_Access","extracurricular_activities","part_time_job","final_grade",
         "enrollment_date_clean","study_time_hours","attendance_percent","sleep_hours",
         "previous_grade","final_exam_score"]
df = df[[c for c in ORDER if c in df.columns]]
df.head()

## 17. Verificación final

In [ ]:
print("Forma final:", df.shape)
print("\nTipos:"); print(df.dtypes)
print("\nNulos por columna:"); print(df.isna().sum())
print("\nRangos numéricos:"); print(df[NUM].describe().loc[["min","max"]])
print("\nDuplicados exactos:", df.duplicated().sum())

## 18. Load — exportar la base limpia

In [ ]:
OUT = "student_performance_clean.csv"
df.to_csv(OUT, index=False)
print("Guardado:", OUT)

try:
    from google.colab import files
    files.download(OUT)
except Exception:
    pass